In [1]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

import numpy as np
import pandas as pd
import torch
import re
from collections import Counter
from transformers import AutoTokenizer, AutoModelForTokenClassification
from transformers import TrainingArguments, Trainer
from sklearn.model_selection import train_test_split
from datasets import Dataset, load_dataset as hf_load_dataset
from sklearn.metrics import f1_score

print("All imports done!")
print("GPU available:", torch.cuda.is_available())

/kaggle/input/competitions/kaz-punct-hackathon/sample_submission.csv
/kaggle/input/competitions/kaz-punct-hackathon/train_example.csv
/kaggle/input/competitions/kaz-punct-hackathon/test.csv
/kaggle/input/datasets/kamilanurzhanova/kaz-dataset/kazakhBooks.csv
/kaggle/input/datasets/kamilanurzhanova/kaz-dataset/oscar.csv
All imports done!
GPU available: True


In [2]:
label2id = {"O": 0, "COMMA": 1, "PERIOD": 2, "QUESTION": 3}
id2label = {v: k for k, v in label2id.items()}
print("Labels:", label2id)

Labels: {'O': 0, 'COMMA': 1, 'PERIOD': 2, 'QUESTION': 3}


In [3]:
train_orig = pd.read_csv('/kaggle/input/competitions/kaz-punct-hackathon/train_example.csv')
test = pd.read_csv('/kaggle/input/competitions/kaz-punct-hackathon/test.csv')

print("Train rows:", len(train_orig))
print("Test rows:", len(test))
print(train_orig.head(2))

Train rows: 500
Test rows: 3552
               id                               input_text          labels
0  kzp_train_0001                    ерден ердің қаупі бар    O O O PERIOD
1  kzp_train_0002  ақылдының ақылы сарқылмайтын көлмен тең  O O O O PERIOD


In [4]:
# Function to strip punctuation and create labels
def strip_and_label(text):
    tokens = text.split()
    labels = []
    clean_tokens = []
    for token in tokens:
        if token.endswith('?'):
            labels.append('QUESTION')
            clean_tokens.append(token[:-1].lower())
        elif token.endswith(('.', '!')):
            labels.append('PERIOD')
            clean_tokens.append(token[:-1].lower())
        elif token.endswith(','):
            labels.append('COMMA')
            clean_tokens.append(token[:-1].lower())
        else:
            labels.append('O')
            clean_tokens.append(token.lower())
    return clean_tokens, labels

# Filter out noisy non-Kazakh sentences
def is_clean(tokens):
    if len(tokens) < 10 or len(tokens) > 60:
        return False
    kazakh_chars = set('абвгдеёжзийклмнопрстуфхцчшщъыьэюяәіңғүұқөһ')
    text = ' '.join(tokens).lower()
    total = len([c for c in text if c.isalpha()])
    kazakh = len([c for c in text if c in kazakh_chars])
    if total == 0:
        return False
    return (kazakh / total) > 0.7

# Stream and collect 50000 clean sentences
hf_dataset = hf_load_dataset(
    "kz-transformers/multidomain-kazakh-dataset",
    split="train",
    streaming=True
)

hf_sentences = []
hf_labels_list = []

print("Loading HuggingFace data...")
for i, example in enumerate(hf_dataset):
    tokens, labs = strip_and_label(example["text"])
    if is_clean(tokens):
        hf_sentences.append(' '.join(tokens))
        hf_labels_list.append(' '.join(labs))
    if len(hf_sentences) >= 50000:
        break
    if i % 10000 == 0:
        print(f"  Processed {i}, collected {len(hf_sentences)} clean sentences")

print(f"Done! Collected {len(hf_sentences)} sentences")


README.md: 0.00B [00:00, ?B/s]

Loading HuggingFace data...
  Processed 0, collected 1 clean sentences
  Processed 10000, collected 4921 clean sentences
  Processed 20000, collected 9603 clean sentences
  Processed 30000, collected 14614 clean sentences
  Processed 40000, collected 18873 clean sentences
  Processed 50000, collected 23326 clean sentences
  Processed 60000, collected 27558 clean sentences
  Processed 70000, collected 31825 clean sentences
  Processed 80000, collected 36940 clean sentences
  Processed 90000, collected 41959 clean sentences
  Processed 100000, collected 46658 clean sentences
Done! Collected 50000 sentences


In [5]:
hf_df = pd.DataFrame({
    'id': [f'hf_{i}' for i in range(len(hf_sentences))],
    'input_text': hf_sentences,
    'labels': hf_labels_list
})

train_full = pd.concat([train_orig, hf_df], ignore_index=True)
train_full = train_full.reset_index(drop=True)

print(f"Total training rows: {len(train_full)}")
question_rows = train_full[
    train_full['labels'].str.contains('QUESTION')
]
print(f"QUESTION sentences: {len(question_rows)}")

# Oversample QUESTION sentences 10x
train_full = pd.concat(
    [train_full] + [question_rows] * 10,
    ignore_index=True
).sample(frac=1, random_state=42).reset_index(drop=True)

# Verify new distribution
all_labels = []
for l in train_full['labels']:
    all_labels.extend(l.split())
print("New distribution:", Counter(all_labels))

# Check label distribution
all_labels = []
for l in train_full['labels']:
    all_labels.extend(l.split())
print("Label distribution:", Counter(all_labels))

Total training rows: 50500
QUESTION sentences: 1843
New distribution: Counter({'O': 1307072, 'PERIOD': 102286, 'COMMA': 90460, 'QUESTION': 25630})
Label distribution: Counter({'O': 1307072, 'PERIOD': 102286, 'COMMA': 90460, 'QUESTION': 25630})


In [6]:
MODEL_NAME = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_and_align(examples):
    words_batch  = [text.split() for text in examples["input_text"]]
    labels_batch = [lbl.split()  for lbl  in examples["labels"]]

    tokenized = tokenizer(
        words_batch,
        is_split_into_words=True,
        truncation=True,
        padding="max_length",
        max_length=128,
    )

    all_labels = []
    for i, label_list in enumerate(labels_batch):
        word_ids = tokenized.word_ids(batch_index=i)
        aligned = []
        prev_word_id = None
        for word_id in word_ids:
            if word_id is None:
                aligned.append(-100)
            elif word_id != prev_word_id:
                aligned.append(label2id[label_list[word_id]])
            else:
                aligned.append(-100)
            prev_word_id = word_id
        all_labels.append(aligned)

    tokenized["labels"] = all_labels
    return tokenized

print("Tokenizer ready!")

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer ready!


In [7]:
train_df, val_df = train_test_split(train_full, test_size=0.1, random_state=42)

train_dataset = Dataset.from_pandas(train_df, preserve_index=False)
val_dataset   = Dataset.from_pandas(val_df,   preserve_index=False)

train_dataset = train_dataset.map(tokenize_and_align, batched=True)
val_dataset   = val_dataset.map(tokenize_and_align,   batched=True)

print(f"Train size: {len(train_dataset)}")
print(f"Val size:   {len(val_dataset)}")

Map:   0%|          | 0/62037 [00:00<?, ? examples/s]

Map:   0%|          | 0/6893 [00:00<?, ? examples/s]

Train size: 62037
Val size:   6893


In [8]:
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=4,
    id2label=id2label,
    label2id=label2id,
)
print("Model loaded!")

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForTokenClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
classifier.bias             | MISSING    | 
classifier.weight           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded!


In [9]:
training_args = TrainingArguments(
    output_dir="./punct-model",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
)

trainer.train()
print("Training done!")

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss
1,0.257196,0.192150
2,0.174430,0.155461
3,0.135761,0.147445
4,0.117886,0.148973
5,0.106819,0.148947


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Training done!


In [10]:
trainer.save_model("./my-punct-model")
tokenizer.save_pretrained("./my-punct-model")
print("Model saved!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved!


In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

def predict(text):
    words = text.lower().split()
    inputs = tokenizer(
        words,
        is_split_into_words=True,
        return_tensors="pt",
        truncation=True,
        max_length=128,
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    predictions = outputs.logits.argmax(dim=-1)[0]
    word_ids = inputs.word_ids()

    result = []
    seen = set()
    for idx, word_id in enumerate(word_ids):
        if word_id is not None and word_id not in seen:
            result.append(id2label[predictions[idx].item()])
            seen.add(word_id)

    return " ".join(result)

test["labels"] = test["input_text"].apply(predict)
submission = test[["id", "labels"]]
submission.to_csv("submission.csv", index=False)
print("Saved! Rows:", len(submission))
print(submission.head(3))

Saved! Rows: 3552
          id                                             labels
0  kzp_00001                      O O O O PERIOD O O O O PERIOD
1  kzp_00002       O O O O O O PERIOD O O O O PERIOD O O PERIOD
2  kzp_00003  O O COMMA O O O O O O O O O PERIOD O O O O O O...
